# MY FARM — Cassava Disease Model Training

Trains an on-device crop-disease classifier on the **Makerere / NaCRRI Cassava Leaf Disease** dataset (21,367 real Ugandan field photos, expert-labelled) and exports a **TensorFlow Lite** model for the Flutter app.

**5 classes:** Cassava Bacterial Blight (CBB), Cassava Brown Streak Disease (CBSD), Cassava Mosaic Disease (CMD), Cassava Green Mottle (CGM), Healthy.

### How to run
1. **Runtime → Change runtime type → GPU** (T4 is fine, free tier).
2. Run every cell top to bottom.
3. You'll need a free **Kaggle account** to download the data (the notebook shows you how).
4. At the end you download `cassava.tflite` and `labels.txt` and drop them into the Flutter app's `assets/models/` folder.

Training on the free T4 GPU takes roughly **30–60 minutes** for the settings below.

## 1. Check the GPU is on

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', gpus if gpus else 'NO GPU — go to Runtime > Change runtime type > GPU')

## 2. Get the dataset from Kaggle

The dataset is the official Kaggle competition **`cassava-leaf-disease-classification`**.

To download it you need a Kaggle API token (free):
1. Go to kaggle.com → your profile → **Account** → **Create New API Token**. A file `kaggle.json` downloads.
2. Run the cell below and upload that `kaggle.json` when prompted.
3. You must also visit the competition page once and click **Join Competition** (accept the rules), or the download will be denied.

In [ ]:
from google.colab import files
import os, json

print('Upload your kaggle.json (Kaggle > Account > Create New API Token)')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!pip -q install kaggle
print('\nKaggle credentials set.')

In [ ]:
# Download (~6 GB) and unzip. Takes a few minutes.
!kaggle competitions download -c cassava-leaf-disease-classification -p /content/data
!cd /content/data && unzip -q -o cassava-leaf-disease-classification.zip -d cassava && echo 'Unzipped.'
!ls /content/data/cassava

## 3. Load and inspect the labels

The competition gives `train.csv` (image filename → numeric label) and `label_num_to_disease_map.json` (numeric label → disease name).

In [ ]:
import pandas as pd, json

DATA = '/content/data/cassava'
IMG_DIR = f'{DATA}/train_images'

df = pd.read_csv(f'{DATA}/train.csv')
with open(f'{DATA}/label_num_to_disease_map.json') as f:
    label_map = json.load(f)              # {'0': 'Cassava Bacterial Blight (CBB)', ...}
label_map = {int(k): v for k, v in label_map.items()}

# IMPORTANT: these label strings must EXACTLY match the keys in the Flutter app's
# treatment_db.dart so each prediction maps to the right treatment text.
# Kaggle numeric order is 0=CBB, 1=CBSD, 2=CGM, 3=CMD, 4=Healthy.
APP_KEY = {
    'Cassava Bacterial Blight (CBB)': 'cassava_bacterial_blight',
    'Cassava Brown Streak Disease (CBSD)': 'cassava_brown_streak_disease',
    'Cassava Green Mottle (CGM)': 'cassava_green_mottle',
    'Cassava Mosaic Disease (CMD)': 'cassava_mosaic_disease',
    'Healthy': 'healthy',
}
class_names = [APP_KEY[label_map[i]] for i in range(len(label_map))]
print('Class labels (must match treatment_db.dart keys):')
for i, n in enumerate(class_names):
    print(f'  {i}: {n}')
print('\nImages per class:')
print(df['label'].map(label_map).value_counts())
df['label'] = df['label'].astype(str)
print('\nTotal images:', len(df))

## 4. Build the data pipeline

We use an 85/15 train/validation split, resize to 224×224 (EfficientNet's native size), and apply light augmentation (flips, rotation, zoom) so the model generalises to new field photos.

The classes are imbalanced (CMD dominates), so we also compute **class weights** to stop the model from simply predicting the majority class.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

IMG_SIZE = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=42)
print(f'Train: {len(train_df)}  Val: {len(val_df)}')

def make_ds(frame, training):
    paths = (IMG_DIR + '/' + frame['image_id']).values
    labels = frame['label'].astype(int).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(2048)
    def load(path, label):
        img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        return img, label
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, True)
val_ds = make_ds(val_df, False)

weights = compute_class_weight('balanced',
    classes=np.arange(len(class_names)),
    y=train_df['label'].astype(int).values)
class_weight = {i: float(w) for i, w in enumerate(weights)}
print('Class weights:', class_weight)

## 5. Build the model (transfer learning)

We start from **EfficientNetB0** pre-trained on ImageNet — a small, fast network well suited to running on a phone — and add our own classification head. Augmentation and EfficientNet's own preprocessing are baked into the model so the exported TFLite file expects plain 0–255 RGB images (simplest possible input for Flutter).

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

data_aug = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='augment')

base = EfficientNetB0(include_top=False, weights='imagenet',
                      input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False   # phase 1: freeze the backbone

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_aug(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
# NOTE: do NOT pass training=False here — that would keep the backbone in
# inference mode even after we unfreeze it in phase 2, crippling fine-tuning.
x = base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 6. Phase 1 — train the new head

In [ ]:
# Phase 1: train ONLY the new classifier head (backbone frozen).
# Give it enough epochs to actually converge before fine-tuning.
cbs1 = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True,
                                     monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3, min_lr=1e-5),
]
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=15,
                  class_weight=class_weight, callbacks=cbs1)

## 7. Phase 2 — fine-tune the top of the backbone

Unfreeze the upper layers and train a little more at a low learning rate. This is where most of the accuracy gain comes from.

In [ ]:
# Phase 2: unfreeze the top of the backbone and fine-tune at a healthy rate.
base.trainable = True
for layer in base.layers[:-60]:   # unfreeze the last ~60 layers
    layer.trainable = False
# keep BatchNorm layers frozen for stable fine-tuning
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),   # 10x higher than before
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

cbs2 = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True,
                                     monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3, min_lr=1e-6),
]
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=15,
                  class_weight=class_weight, callbacks=cbs2)

## 8. Evaluate — confusion matrix & per-class report
These numbers go straight into your report.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np, matplotlib.pyplot as plt

y_true, y_pred = [], []
for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right'); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, cm[i, j], ha='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=120); plt.show()
print('Saved confusion_matrix.png (use it in your report)')

## 9. Export to TensorFlow Lite

We export two things the Flutter app needs: the quantised `.tflite` model (small, fast on phones) and a `labels.txt` listing the class names in order.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # dynamic-range quantisation
tflite_model = converter.convert()

# Filenames MUST match what InferenceService loads: <crop>.tflite + <crop>_labels.txt
with open('cassava.tflite', 'wb') as f:
    f.write(tflite_model)
with open('cassava_labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

size_mb = len(tflite_model) / 1e6
print(f'cassava.tflite  ({size_mb:.1f} MB)')
print('cassava_labels.txt:'); print('\n'.join(class_names))

## 10. Quick sanity test of the exported TFLite model

In [ ]:
interp = tf.lite.Interpreter(model_path='cassava.tflite')
interp.allocate_tensors()
inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
print('Input shape:', inp['shape'], inp['dtype'])
print('Output shape:', out['shape'])

# run one validation image through it
for imgs, labels in val_ds.take(1):
    one = imgs[0:1].numpy().astype(np.float32)
    interp.set_tensor(inp['index'], one)
    interp.invoke()
    probs = interp.get_tensor(out['index'])[0]
    print('Predicted:', class_names[int(np.argmax(probs))],
          f'({100*np.max(probs):.1f}%)  |  Actual:', class_names[int(labels[0])])

## 11. Download the files for the Flutter app

In [ ]:
from google.colab import files
files.download('cassava.tflite')
files.download('cassava_labels.txt')
files.download('confusion_matrix.png')
print('Put cassava.tflite and labels.txt into:  myfarm_flutter/assets/models/')